# Step 5 – Restrict observation window
**Purpose:** keep publications 1950–2023 with AID ≠ 0, compute the first publication year per AID and keep authors with entry < 2019.

**Input:** `Final_Result.selected.csv` (step 4). **Outputs:** `Final_Result_from1950s.selected.parquet`, `filtered_data.csv`
(author × paper rows; gender is added in step 10), `filtered_data_topicnames.csv`.

**Note:** section 1 creates the parquet file and only needs to be run once; if it exists, start at section 2.
Topic names come from the OpenAlex topic mapping table (`data/raw/openalex_topic_field_subfield_table.csv`).


## 1. Create the 1950–2023 parquet file (run once)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import duckdb
import os

BASE_DIR = os.path.abspath('..')                  # root of the replication package (notebooks run from code/)
RAW_DIR = f'{BASE_DIR}/data/raw'                   # PKG / MEDLINE / OpenAlex input files (not included)
DERIVED_DIR = f'{BASE_DIR}/data/derived'           # intermediate files written by steps 1-11
os.makedirs(DERIVED_DIR, exist_ok=True)

import data from all papers in topics with good coverage

In [ ]:
#import
df_full = pd.read_csv(f'{DERIVED_DIR}/Final_Result.selected.csv')

In [ ]:
df_full.columns

delete irrelevant topics, remove begin_year as it is unreliable, remove 2025 and 2024 as they are incomplete, remove AIDs that are 0 because they obscure the begin_year calculation

In [ ]:
duckdb.sql(f'''
    copy (
        select
           id, PMID, PubYear, AID, topics_num, display_name, share_outside_PubMed_per_topic
        from read_csv(
            '{DERIVED_DIR}/Final_Result.selected.csv',
            header=true,
            delim=',',
            quote='"',
            escape='"',
            sample_size=-1
        )
        where PubYear < 2024 AND PubYear > 1949 AND AID <> 0 AND AID IS NOT NULL
    )
    to '{DERIVED_DIR}/Final_Result_from1950s.selected.parquet'
    (format parquet)
''')

## 2. Entry year and restriction to entry before 2019

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import duckdb
import os

BASE_DIR = os.path.abspath('..')                  # root of the replication package (notebooks run from code/)
RAW_DIR = f'{BASE_DIR}/data/raw'                   # PKG / MEDLINE / OpenAlex input files (not included)
DERIVED_DIR = f'{BASE_DIR}/data/derived'           # intermediate files written by steps 1-11
os.makedirs(DERIVED_DIR, exist_ok=True)

In [ ]:
con = duckdb.connect(f'{DERIVED_DIR}/restrict_period.duckdb')

In [ ]:
con.execute(f"""
    CREATE OR REPLACE TABLE df_full AS 
    SELECT * FROM read_parquet('{DERIVED_DIR}/Final_Result_from1950s.selected.parquet')
""")

## calculate begin_year from first publication and select only begin_years before 2019

In [ ]:
con.execute("""
    CREATE OR REPLACE TEMP TABLE author_first_pub AS
    SELECT
        AID,
        MIN(PubYear) AS begin_year
    FROM df_full
    WHERE AID IS NOT NULL
      AND PubYear IS NOT NULL
    GROUP BY AID
""")

In [ ]:
con.execute("""
    CREATE OR REPLACE TEMP TABLE author_df AS
    SELECT
        a.*,
        p.begin_year
    FROM df_full a
    LEFT JOIN author_first_pub p
        ON a.AID = p.AID
    WHERE p.begin_year < 2019
""")

In [ ]:
author_df = con.execute("SELECT * FROM author_df, limit 30").df()

In [ ]:
author_df

In [ ]:
df_selected = con.execute("""
    SELECT
        AID,
        PMID,
        begin_year,
        topics_num,
        PubYear
    FROM author_df
""").df()

In [ ]:
df_selected = df_selected.fillna('UNK') #unify missings

In [ ]:
df_selected.to_csv(f'{DERIVED_DIR}/filtered_data.csv', index = False)

In [ ]:
sorted(df_selected['begin_year'].unique()) #sanity check

# Topic information

As the data only contains numbers representing topics, the topics names first have to be merged to the data.

In [ ]:
topicnames = pd.read_csv(f'{RAW_DIR}/openalex_topic_field_subfield_table.csv')

In [ ]:
topicnames.columns

In [ ]:
df_selected.columns

In [ ]:
topic_mapping = topicnames.set_index('topic_id')['topic_name'].to_dict()

In [ ]:
df_selected['topic_name'] = df_selected.topics_num.map(topic_mapping)
df_selected['topic_name'].fillna('UNK', inplace=True)

In [ ]:
df_selected.head() #investigate results

In [ ]:
df_selected.to_csv(f'{DERIVED_DIR}/filtered_data_topicnames.csv', index = False)